# 🎨 Projeto da Unidade 1

**Instituto Federal da Paraíba — Campus Cajazeiras**

Curso: **Análise e Desenvolvimento de Sistemas**

Disciplina: **Processamento de Imagem e Visão Computacional**

Docente: **Prof.ª Dra. Kyury Barros**

---

Neste projeto, vocês vão aplicar os conteúdos da **Unidade 1** (representação de imagens como arrays, tipos de dados, regiões de interesse, amostragem, interpolação, quantização, canais de cor e transparência) em um conjunto de manipulações de imagens com finalidade prática: simulação de dispositivos de exibição, construção de interfaces gráficas, composição de imagens com transparência, efeitos ópticos, simetria e ocultação de informação.

### 🗺️ Mapa do projeto
O projeto está organizado em **4 etapas obrigatórias**, com **12 tarefas**, e uma **etapa desafio** opcional, com 2 tarefas. Recomenda-se seguir a ordem proposta: em cada etapa, as primeiras tarefas fornecem a base técnica para as seguintes.

| Etapa | Tarefas | Conteúdo |
|---|---|---|
| 🎮 1 — Estúdio de Games | 1.1 a 1.3 | simulação de telas de consoles antigos e de uma tela de RPG |
| 😂 2 — Memes e Figurinhas | 2.1 a 2.3 | inserção de texto, ampliação de regiões e imagens com transparência |
| 🏘️ 3 — Simulador de miniatura | 3.1 a 3.3 | máscaras de foco e desfoque por reamostragem (*tilt-shift*) |
| 🔷 4 — Caleidoscópio e simetria | 4.1 a 4.3 | transformações geométricas por fatiamento, transposição e coordenadas|
| 🕵️ Desafio — Esteganografia | 5.1 a 5.2 | ocultação de uma imagem nos bits menos significativos de outra (opcional) |

### 📌 Como trabalhar neste notebook
1. Faça uma cópia: **Arquivo → Salvar uma cópia no Drive**.
2. Preencha a identificação da equipe logo abaixo (poderá ser feito em dupla ou individual).
3. Execute a **Parte 0** (configuração do ambiente e envio da foto principal).
4. Desenvolva as etapas na ordem proposta. Os **exemplos resolvidos** (✨) servem como referência de estilo e precisam apenas ser executados.
5. Tarefas ainda não implementadas não interrompem a execução do notebook; é possível avançar e retornar a elas depois.
6. Antes da entrega, execute **Ambiente de execução → Reiniciar sessão e executar tudo**. O notebook deve rodar do início ao fim sem erros.

### 🧩 Como cada etapa e cada tarefa estão organizadas
Cada etapa começa com uma apresentação do tema e um **exemplo resolvido** (✨), que deve ser executado e lido com atenção: ele introduz a técnica que as tarefas da etapa vão desenvolver.

Cada célula de código contém, nesta ordem: (1) a linha 📤 `ENVIAR = False`, que permite usar **outra imagem** apenas naquela tarefa se desejar alterando para `True`; (2) a função a ser implementada no lugar de `# SEU CÓDIGO AQUI`; (3) a chamada que executa a função e exibe entrada e saída lado a lado (⏳ indica tarefa ainda não implementada).

## 👥 Identificação da equipe
| Campo | Preencha |
|---|---|
| Integrante 1 | |
| Integrante 2 | |

---
# Parte 0 — Configuração (execute antes de tudo)
### 0.1 Importações e funções auxiliares
As duas células abaixo preparam o ambiente e definem as funções de apoio `mostrar`, `mostrar_varias`, `mostrar_resultado` (executa uma tarefa e exibe entrada e saída lado a lado) e `salvar_resultado`.

In [ ]:
import os
import glob
import urllib.request

import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data as sk_data

try:
    import google.colab  # só existe dentro do Colab
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

plt.rcParams["figure.dpi"] = 100
os.makedirs("resultados", exist_ok=True)

print("OpenCV:", cv2.__version__, "| NumPy:", np.__version__)
print("Rodando no Google Colab ✅" if EM_COLAB else "Rodando fora do Colab")

In [ ]:
def mostrar(img, titulo=None, bgr=True, ax=None, tamanho=6, eixos=False):
    """Exibe uma imagem com Matplotlib (cinza, BGR ou BGRA)."""
    sozinha = ax is None
    if sozinha:
        _, ax = plt.subplots(figsize=(tamanho, tamanho))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255) if img.dtype == np.uint8 else ax.imshow(img, cmap="gray")
    else:
        if bgr and img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        elif bgr and img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA)
        ax.imshow(img)
    if titulo:
        ax.set_title(titulo)
    if not eixos:
        ax.axis("off")
    if sozinha:
        plt.show()


def mostrar_varias(imagens, titulos=None, bgr=True, colunas=3, tamanho=4, eixos=False):
    """Exibe várias imagens lado a lado em uma grade."""
    n = len(imagens)
    colunas = min(colunas, n)
    linhas = int(np.ceil(n / colunas))
    fig, axs = plt.subplots(linhas, colunas, figsize=(tamanho * colunas, tamanho * linhas), squeeze=False)
    for k, ax in enumerate(axs.flat):
        if k < n:
            mostrar(imagens[k], titulos[k] if titulos else None, bgr=bgr, ax=ax, eixos=eixos)
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def limitar_tamanho(img, maior_lado=1024):
    """Reduz a imagem para que o maior lado tenha no máximo `maior_lado` pixels."""
    h, w = img.shape[:2]
    escala = maior_lado / max(h, w)
    if escala < 1:
        img = cv2.resize(img, (int(w * escala), int(h * escala)), interpolation=cv2.INTER_AREA)
    return img


def mostrar_resultado(funcao, *args, **kwargs):
    """Executa uma função das tarefas e mostra a entrada e a saída lado a lado.

    Se a tarefa ainda não foi feita (NotImplementedError), só avisa com ⏳, para o notebook
    continuar rodando. Atenção: ela NÃO confere se a resposta está certa; para isso,
    use o "✅ Confira" de cada tarefa.
    Retorna a saída da função (ou None).
    """
    try:
        saida = funcao(*args, **kwargs)
    except NotImplementedError:
        print(f"⏳ {funcao.__name__}: ainda não implementado.")
        return None
    entrada = next((a for a in args if isinstance(a, np.ndarray)), None)
    if isinstance(saida, np.ndarray):
        if entrada is not None and entrada.ndim in (2, 3):
            mostrar_varias([entrada, saida], ["entrada", funcao.__name__], tamanho=5)
        else:
            mostrar(saida, funcao.__name__)
    return saida

def salvar_resultado(img, nome):
    """Salva a imagem em resultados/<nome> e informa o tamanho do arquivo."""
    caminho = os.path.join("resultados", nome)
    ok = cv2.imwrite(caminho, img)
    print(("💾 salvo: " if ok else "❌ falha ao salvar: ") + caminho +
          (f" ({os.path.getsize(caminho) / 1024:.1f} KB)" if ok else ""))
    return ok


print("Funções auxiliares prontas ✅")

### 📸 Foto principal
A célula abaixo define a função de envio de imagens (utilizada pela linha 📤 de cada tarefa) e carrega a **foto principal**, usada por padrão em todas as tarefas.
- **Na primeira execução**, a seleção de arquivo abre automaticamente: escolha uma foto da equipe. Ela fica armazenada e não será solicitada novamente.
- Para **trocar** a foto principal, altere `ENVIAR = False` para `True`, execute a célula e retorne para `False`.
- Em cada tarefa, a linha 📤 funciona da mesma forma: com `ENVIAR = True`, a tarefa passa a usar outra imagem, também armazenada.
- Se nenhum arquivo for selecionado, é utilizada uma imagem de exemplo.

In [ ]:
def imagem_da_tarefa(tarefa, enviar=False, padrao=None, reduzir=True, usar_foto_teste=True,
                      pedir_se_vazio=False):
    """Carrega a imagem que será usada em uma tarefa.

    Parâmetros
    ----------
    tarefa : str
        Identificador da tarefa (ex.: "1.3"). A imagem enviada é salva em
        imagens_tarefas/ com esse nome, para ser reutilizada nas próximas execuções.
    enviar : bool
        True abre a seleção de arquivo (botão no Colab ou janela no computador)
        e substitui a imagem salva da tarefa. Após o envio, volte para False.
    padrao : ndarray ou None
        Imagem usada quando nenhuma foi enviada para a tarefa.
    reduzir : bool
        True reduz imagens grandes para no máximo 1024 px no maior lado.
        Use False quando os valores dos pixels precisam ser preservados
        exatamente, como na esteganografia.
    usar_foto_teste : bool
        Se True e `padrao` for None, usa a foto principal (img_teste).
        Se False, devolve `padrao` mesmo que seja None.
    pedir_se_vazio : bool
        Se True e ainda não houver imagem salva para a tarefa, abre a seleção
        de arquivo automaticamente (usado na foto principal).

    Ordem de escolha da imagem
    --------------------------
    1. a imagem enviada para a tarefa (se existir);
    2. a imagem `padrao`;
    3. a foto principal (img_teste), se `usar_foto_teste=True`;
    4. uma imagem de exemplo, se nenhuma foto principal foi enviada.

    Retorna
    -------
    ndarray BGR uint8 (ou None, se `usar_foto_teste=False` e `padrao=None`).
    """
    pasta = "imagens_tarefas"
    os.makedirs(pasta, exist_ok=True)
    base = "tarefa_" + tarefa.replace(".", "_")
    if pedir_se_vazio and not glob.glob(os.path.join(pasta, base + ".*")):
        print(f"📤 Nenhuma imagem salva para '{tarefa}': selecione o arquivo.")
        enviar = True
    if enviar:
        for antigo in glob.glob(os.path.join(pasta, base + ".*")):
            os.remove(antigo)
        try:
            if EM_COLAB:
                from google.colab import files
                enviados = files.upload()
                if enviados:
                    nome, conteudo = next(iter(enviados.items()))
                    with open(os.path.join(pasta, base + os.path.splitext(nome)[1].lower()), "wb") as f:
                        f.write(conteudo)
            else:
                import shutil
                import tkinter as tk
                from tkinter import filedialog
                janela = tk.Tk()
                janela.withdraw()
                janela.attributes("-topmost", True)
                caminho = filedialog.askopenfilename(
                    title=f"Imagem para a tarefa {tarefa}",
                    filetypes=[("Imagens", "*.jpg *.jpeg *.png *.bmp *.webp"), ("Todos", "*.*")])
                janela.destroy()
                if caminho:
                    shutil.copy(caminho, os.path.join(pasta, base + os.path.splitext(caminho)[1].lower()))
        except Exception as erro:
            print("Não foi possível abrir a seleção de arquivo:", erro)
            print(f"👉 Copie a imagem para {os.path.abspath(pasta)} com o nome {base}.jpg (ou .png)")
    guardadas = sorted(glob.glob(os.path.join(pasta, base + ".*")))
    if guardadas:
        img = cv2.imread(guardadas[0])
        if img is not None:
            img = limitar_tamanho(img) if reduzir else img
            print(f"📷 tarefa {tarefa}: usando a imagem enviada ({img.shape[1]}x{img.shape[0]})")
            return img
        print("⚠️ arquivo guardado não é uma imagem:", guardadas[0])
    if padrao is None and usar_foto_teste:
        if "img_teste" in globals():
            return img_teste
        print("Nenhuma foto principal enviada: usando uma imagem de exemplo. Para enviar a sua, use ENVIAR = True na célula da foto principal.")
        return cv2.cvtColor(sk_data.chelsea(), cv2.COLOR_RGB2BGR)
    return padrao

# 📸 FOTO PRINCIPAL: na primeira execução, a seleção de arquivo abre automaticamente.
#    Para trocar a foto depois, altere ENVIAR para True, execute e retorne para False.
ENVIAR = False
img_teste = imagem_da_tarefa("principal", ENVIAR, pedir_se_vazio=True)
mostrar(img_teste, "foto principal")

---

# 🎮 Etapa 1 — Estúdio de Games
Nesta etapa, vocês vão reproduzir técnicas de imagem usadas em videogames antigos. Nas duas primeiras tarefas, simulam as **limitações de hardware** dos consoles da época: os 4 tons da tela do **Game Boy** e a baixa resolução com poucas cores dos consoles **8-bit**, aplicando **quantização** e **reamostragem**. Na terceira, reaproveitam o filtro 8-bit para compor uma **tela de RPG** com uma caixa de diálogo, trabalhando com **fatiamento**, **regiões de interesse** e **desenho de formas e texto**.

### ✨ Exemplo resolvido — Monitor de fósforo verde
Os primeiros monitores de computador eram **monocromáticos**: a tela era revestida de fósforo verde, e cada ponto podia apenas variar de **intensidade**, sem cor própria. Simular esse dispositivo é, portanto, exibir a **luminância** da imagem usando somente o canal verde.

A implementação segue três passos:
1. **Converter para tons de cinza** (`cv2.cvtColor`): o resultado é um array 2D `(h, w)` com a intensidade de cada pixel, que é exatamente a informação que o monitor monocromático consegue exibir.
2. **Criar uma imagem preta** com o mesmo formato e tipo da entrada (`np.zeros_like`): um array `(h, w, 3)` em `uint8`, com os três canais zerados. Como é um array **novo**, a imagem original não é alterada.
3. **Copiar a intensidade para o canal G** (`saida[:, :, 1] = cinza`): lembrando da ordem **BGR** do OpenCV, o índice 1 é o verde. Os canais B (0) e R (2) permanecem em zero, de modo que cada pixel varia apenas do preto ao verde puro.

> 📚 Conceitos da *Aula Prática 01*: conversão para tons de cinza (Parte 4), arrays 3D e canais de cor (Partes 3.4 e 7.1), criação de imagens do zero (Parte 5.6) e a ordem BGR (Parte 4.2).

Este exemplo também mostra o **padrão esperado nas funções do projeto**:
- **docstring** (o texto entre aspas triplas `"""` logo abaixo do `def`), descrevendo o que a função faz e os conceitos utilizados. Diferente dos comentários (`#`), a docstring fica associada à função e pode ser consultada com `help(nome_da_funcao)`;
- a **imagem de entrada nunca é modificada**: a função trabalha sobre um array novo ou sobre uma cópia (`.copy()`);
- operações **vetorizadas**, sem laços percorrendo os pixels;
- o retorno é uma nova imagem **BGR `uint8`**.

In [ ]:
def filtro_fosforo_verde(img):
    """Simula um monitor monocromático de fósforo verde.

    Conceitos: conversão para cinza, canais de cor, criação de imagem do zero.
    """
    cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    saida = np.zeros_like(img)          # nova imagem preta, mesmo shape e dtype
    saida[:, :, 1] = cinza              # a intensidade vai para o canal G
    return saida

_ = mostrar_resultado(filtro_fosforo_verde, img_teste)

### ✍️ Tarefa 1.1 — Game Boy
🎯 **Objetivo:** simular a tela do Game Boy original, que exibia apenas **4 tons de verde**.

<p align="center">
  <img src="https://preview.redd.it/love-my-new-old-backlit-dmg-first-game-on-the-list-pok%C3%A9mon-v0-pbdj1pn34iy51.jpg?width=1080&crop=smart&auto=webp&s=c4fa5973ad419ba97dadac29ba35e1597bb27ac2" width="320" alt="Game Boy original">
</p>

**Por que apenas 4 tons?** A tela do Game Boy original (1989) armazenava cada pixel com apenas **2 bits**. Com 2 bits, existem 2² = **4 combinações possíveis** (`00`, `01`, `10`, `11`), e cada uma corresponde a um tom, do mais escuro ao mais claro. Ou seja, a tela é como se fosse uma imagem **quantizada em 2 bits**.

**Partindo do exemplo resolvido:** o monitor de fósforo verde já exibe a intensidade da imagem no canal verde, com **256 níveis**. O Game Boy é o mesmo princípio, mas com apenas **4 níveis**. Basta acrescentar uma etapa ao exemplo para **quantizar** o cinza antes de copiá-lo para o canal G.

In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_1_1 = imagem_da_tarefa("1.1", ENVIAR, padrao=img_teste)

def filtro_gameboy(img):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(filtro_gameboy, img_1_1)

### ✍️ Tarefa 1.2 — Consoles de 8-bit
🎯 **Objetivo:** simular a saída gráfica dos consoles 8-bit, com **pixels grandes** e **poucas cores**.

<p align="center">
  <img src="https://i.pinimg.com/736x/97/0d/54/970d545bd3d92f30ee38affde326794a.jpg" width="220" alt="Game Boy original">
</p>

**Por que pixels grandes e poucas cores?** Os consoles 8-bit dos anos 1980 tinham pouca memória de vídeo. Para caber nela, a imagem era gerada em **baixa resolução** (cada ponto da tela ocupava um bloco de vários pixels) e com **poucos níveis por canal de cor**. São duas limitações diferentes, e cada uma corresponde a um conceito:
- **Resolução espacial** (quantos pixels a imagem tem): para obter pixels grandes, a imagem é **reduzida** e depois **ampliada de volta** ao tamanho original. Por exemplo, uma imagem de 512 × 512 reduzida por um fator 8 fica com 64 × 64 pixels, nessa redução os detalhes finos se perdem. Ao ampliá-la de volta para 512 × 512 com **vizinho mais próximo** (`INTER_NEAREST`), cada um desses 64 × 64 pixels é repetido num bloco de 8 × 8, e é isso que produz o aspecto quadriculado. Para a redução, use `INTER_AREA`, que calcula a média de cada região.
- **Resolução de intensidade** (quantos valores cada pixel pode assumir): numa imagem comum, cada canal (B, G e R) tem 8 bits, ou seja, 256 níveis, o que permite cerca de 16,7 milhões de cores. **Quantizar** cada canal com menos bits reduz esse número, sendo que com $k$ bits cada canal passa a ter $L = 2^k$ níveis. Com 2 bits por canal, por exemplo, cada canal tem apenas 4 níveis, e a imagem passa a ter no máximo 4 × 4 × 4 = **64 cores**. É a mesma quantização da Tarefa 1.1, agora aplicada aos três canais da imagem colorida.

**Partindo da Tarefa 1.1:** no Game Boy, vocês quantizaram **um único canal** (o cinza). Aqui a quantização é aplicada aos **três canais** da imagem colorida, e antes dela entra a redução de resolução. Na função `filtro_8bit()`, o parâmetro `fator` controla o tamanho dos blocos e `bits` o número de níveis por canal. A saída deve ter as **mesmas dimensões** da entrada.

In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_1_2 = imagem_da_tarefa("1.2", ENVIAR, padrao=img_teste)

def filtro_8bit(img, fator=8, bits=2):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(filtro_8bit, img_1_2)

### ✍️ Tarefa 1.3 — Caixa de diálogo de RPG
🎯 **Objetivo:** criar uma tela de RPG com a imagem da tarefa anterior e uma **caixa de diálogo** com uma fala do personagem.

<p align="center">
  <img src="https://static.wikia.nocookie.net/essentialsdocs/images/3/3a/ChoosingStarter.png/revision/latest?cb=20190219205247" width="420" alt="Game Boy original">
</p>

**Como é a tela?** Nos RPGs clássicos, as falas dos personagens aparecem numa caixa na parte inferior da tela, desenhada **por cima** da cena do jogo. A tela desta tarefa contém a cena em pixel art e, na parte inferior, uma caixa de cor sólida, com borda e o `texto` da fala.

**Partindo da Tarefa 1.2:** o aspecto de pixel art da cena vem do seu `filtro_8bit`. A caixa é uma **região de interesse** pintada com uma única cor por atribuição a uma fatia. A borda e o texto são desenhados com `cv2.rectangle` e `cv2.putText`, lembrando que essas funções usam coordenadas **(x, y)**, ao contrário da indexação `img[y, x]`.

In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_1_3 = imagem_da_tarefa("1.3", ENVIAR, padrao=img_teste)

def caixa_de_dialogo(img, texto="SAUDACOES, AVENTUREIRO! (use sua criatividade)"):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(caixa_de_dialogo, img_1_3)

---
# 😂 Etapa 2 — Memes e Figurinhas
Nesta etapa, vocês vão implementar operações comuns em aplicativos de edição para redes sociais como inserção de **texto** posicionado de forma proporcional à imagem, ampliação progressiva de uma **região de interesse** e criação de **imagens com transparência** (canal alfa) no formato de figurinhas para aplicativos de mensagem.

### ✨ Exemplo resolvido — Legenda de tirinha
Nas tirinhas e em muitos memes, o texto fica numa **faixa preta/branca abaixo da imagem**, e não sobre ela. Assim, o texto não esconde nenhuma parte da foto e fica sempre legível.

A implementação segue quatro passos:

1. **Criar a faixa preta.** É uma imagem nova, feita com `np.zeros`, com a **mesma largura** da foto. A altura da faixa costuma ser **1/7 da altura da foto**, assim uma foto com 700 pixels de altura, por exemplo, recebe uma faixa de 100 pixels.

2. **Escolher o tamanho da letra.** No `putText`, o tamanho da letra é definido pela **escala**. Para que o texto acompanhe o tamanho da foto, a escala inicial é `w / 600`, ou seja, vale 1 numa foto de 600 pixels de largura, 2 numa de 1200 pixels, e assim por diante. Em seguida, `cv2.getTextSize` **mede** a largura e a altura que o texto vai ocupar, em pixels. Se o texto for mais largo que a faixa, o laço `while` diminui a escala aos poucos e mede de novo, até o texto caber.

3. **Escrever o texto no centro da faixa.**
   - Na **horizontal**: o espaço que sobra (largura da faixa menos largura do texto) é dividido igualmente entre os dois lados: `x = (largura_da_faixa - largura_do_texto) // 2`.
   - Na **vertical**: o `y` do `putText` indica a **base** das letras, e não o topo. Por isso, para centralizar, usa-se `y = (altura_da_faixa + altura_do_texto) // 2`.

4. **Juntar a foto e a faixa.** `np.vstack` coloca a faixa embaixo da foto. Isso só funciona porque as duas imagens têm a mesma largura e o mesmo número de canais.

> 📚 Conceitos da *Aula Prática 01*: criação de imagens do zero (Parte 5.6), escrita com `putText` (Parte 5.5) e concatenação de arrays (Exercício 5.2).

Medir e centralizar texto são operações que vocês vão usar em todas as tarefas desta etapa.

In [ ]:
def faixa_de_legenda(img, texto="o poder dos memes esta com voce"):
    """Acrescenta uma faixa preta com legenda embaixo da imagem.

    Conceitos: imagem criada do zero, np.vstack (empilhar), putText com getTextSize para centralizar.
    """
    texto = texto.upper()   # converte todas as letras do texto em maiúsculas
    h, w = img.shape[:2]
    faixa = np.zeros((max(50, h // 7), w, 3), dtype=np.uint8)
    fonte, escala = cv2.FONT_HERSHEY_DUPLEX, w / 600
    while cv2.getTextSize(texto, fonte, escala, 2)[0][0] > w - 20:    # diminui a letra até caber
        escala -= 0.05
    (tw, th), _ = cv2.getTextSize(texto, fonte, escala, 2)
    y = (faixa.shape[0] + th) // 2                                   # centraliza na vertical
    cv2.putText(faixa, texto, ((w - tw) // 2, y), fonte, escala, (255, 255, 255), 2, cv2.LINE_AA)
    return np.vstack([img, faixa])

_ = mostrar_resultado(faixa_de_legenda, img_teste)

### ✍️ Tarefa 2.1 — Meme clássico
🎯 **Objetivo:** inserir texto branco com contorno preto no **topo** e na **base** da imagem, no formato tradicional de meme.

<p align="center">
  <img src="https://media.licdn.com/dms/image/v2/D4D22AQH2pPNucVHkag/feedshare-shrink_800/feedshare-shrink_800/0/1710429020395?e=2147483647&v=beta&t=GFXziJjoXdP0GM1nwUnwH0K-gbsYIG7-kIKkodM0J1E" width="420" alt="Game Boy original">
</p>

**Qual é o desafio?** As fotos têm tamanhos muito diferentes, e um texto que fica bom numa imagem de 400 px some numa de 3000 px. Por isso, o tamanho e a posição do texto devem ser calculados **a partir do `shape`** da imagem, e não com valores fixos. O contorno preto garante que o texto seja legível sobre qualquer fundo.

**Partindo do exemplo resolvido:** a legenda de tirinha já mostra como medir, ajustar e centralizar um texto com `cv2.getTextSize`. A diferença é que aqui o texto é escrito **sobre a própria foto**, e não numa faixa separada. O contorno é feito escrevendo o mesmo texto duas vezes, na mesma posição. Primeiro em preto, com uma espessura maior, e depois em branco, com uma espessura menor por cima. A parte preta que sobra ao redor das letras forma o contorno.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_2_1 = imagem_da_tarefa("2.1", ENVIAR, padrao=img_teste)

def meme_classico(img, topo="QUANDO SAI O RESULTADO", base="EU VENDO MINHA NOTA"):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(meme_classico, img_2_1)

### ✍️ Tarefa 2.2 — Zoom dramático
🎯 **Objetivo:** montar um painel com quadros lado a lado, cada um mostrando uma **ampliação** maior em torno de um mesmo ponto da imagem.

<p align="center">
  <img src="https://i.ibb.co/TfQZYQj/zoom.png" width="420" alt="Game Boy original">
</p>

**Como funciona o painel?** Cada quadro é um **recorte quadrado** centrado no ponto escolhido, ampliado para `tamanho × tamanho`. De um quadro para o seguinte, o recorte cai pela metade e a ampliação dobra. Como nenhuma interpolação recupera detalhes perdidos, o último quadro fica visivelmente menos nítido. Se `centro` for `None`, usa-se o centro da imagem.

**Partindo do exemplo resolvido:** a legenda de tirinha juntou duas imagens com `np.vstack`. Aqui os quadros são unidos lado a lado com `np.hstack`. Cada recorte é uma **região de interesse** e a ampliação usa interpolação. Se o recorte ultrapassar a borda da imagem, o fatiamento devolve menos pixels sem gerar erro, então o canto do recorte deve ser mantido dentro da imagem.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_2_2 = imagem_da_tarefa("2.2", ENVIAR, padrao=img_teste)

def zoom_dramatico(img, centro=None, niveis=3, tamanho=300):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(zoom_dramatico, img_2_2)

### ✍️ Tarefa 2.3 — Figurinha
🎯 **Objetivo:** criar uma figurinha para aplicativos de mensagem: a foto recortada em **círculo com borda suave**, fundo **transparente** e uma **legenda** com contorno na parte inferior.

<p align="center">
  <img src="https://i.ibb.co/35b3dLyh/dev-2.png" width="220" alt="Game Boy original">
</p>

**Como uma imagem tem partes transparentes?** Pelo **canal alfa**, um 4º canal além de B, G e R que indica a opacidade de cada pixel (0 é transparente e 255 é opaco). A figurinha é uma imagem **BGRA** de `lado × lado` em que o alfa vale 255 dentro do círculo, 0 fora dele e forma uma rampa na borda, para que o recorte não fique serrilhado. Numa imagem BGRA, as cores do `putText` também têm 4 valores, e com `A = 255` a legenda fica opaca inclusive fora do círculo.

**Partindo da Tarefa 2.1:** a legenda segue a mesma lógica do meme. Para o recorte circular, `np.mgrid` gera as coordenadas de todos os pixels e, com elas, calcula-se a distância de cada pixel ao centro. Pixels a uma distância menor que o raio ficam opacos, e os demais ficam transparentes. Antes de tudo, a imagem precisa ficar quadrada, e para isso completem a função auxiliar `recorte_quadrado(img, lado)`, que recorta o **maior quadrado central** e o redimensiona para `lado × lado`. Ela será reaproveitada na Etapa 4.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_2_3 = imagem_da_tarefa("2.3", ENVIAR, padrao=img_teste)

def recorte_quadrado(img, lado=512):
    # SEU CÓDIGO AQUI
    raise NotImplementedError


def figurinha(img, texto="DEV SENIOR (use a criatividade)", lado=512):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

FIGURINHA = mostrar_resultado(figurinha, img_2_3)

In [ ]:
# 📲 Salvar a figurinha para o WhatsApp (512 x 512, WebP com transparência)
try:
    if FIGURINHA is not None:
        cv2.imwrite("resultados/minha_figurinha.webp", FIGURINHA, [cv2.IMWRITE_WEBP_QUALITY, 80])
        lida = cv2.imread("resultados/minha_figurinha.webp", cv2.IMREAD_UNCHANGED)
        print("figurinha salva:", lida.shape, f"{os.path.getsize('resultados/minha_figurinha.webp') / 1024:.0f} KB")
        if EM_COLAB:
            from google.colab import files
            files.download("resultados/minha_figurinha.webp")
except NameError:
    print("⏳ termine a Tarefa 2.3 para salvar a figurinha.")

---
# 🏘️ Etapa 3 — Simulador de miniatura (*tilt-shift*)
O efeito *tilt-shift* faz cenas reais parecerem **maquetes em miniatura**. Ele reproduz a pequena profundidade de campo das fotografias macro: apenas uma **região permanece nítida**, o restante é desfocado gradualmente, e as cores são intensificadas.

Nesta etapa, o desfoque deve ser obtido **apenas por amostragem e interpolação**, sem o uso de funções de filtragem como `GaussianBlur`.

**Imagem recomendada:** fotografia tomada de um ponto **elevado**, com vista inclinada para baixo, contendo pessoas, veículos ou edificações em escala reduzida.

### ✨ Exemplo resolvido — Contraste
O **contraste** é a diferença entre as regiões claras e escuras de uma imagem. Aumentá-lo significa **afastar** cada valor do nível médio (128): os escuros ficam mais escuros e os claros, mais claros.

A operação é uma única expressão vetorizada: `128 + c * (img - 128)`.
- Com `c > 1`, os valores se afastam de 128 e o contraste **aumenta**; com `c < 1`, eles se aproximam e o contraste **diminui**; com `c = 1`, a imagem não muda.
- A conta é feita em **`float`** e o resultado passa por **`np.clip(..., 0, 255)`** antes de voltar para `uint8`. Sem isso, os valores que passam de 255 ou ficam abaixo de 0 "dariam a volta" (*overflow*).

> 📚 Conceitos da *Aula Prática 01*: tipo `uint8` e *overflow* (Parte 3.2) e operações vetorizadas (Parte 3.5).

Esta função será utilizada na Tarefa 3.3.

In [ ]:
def contraste(img, c=1.3):
    """Aumenta (c > 1) ou diminui (c < 1) o contraste em torno de 128.

    Conceitos: conversão para float, operação vetorizada, clip para evitar overflow.
    """
    return np.clip(128 + c * (img.astype(np.float32) - 128), 0, 255).astype(np.uint8)

mostrar_varias([contraste(img_teste, c) for c in (0.5, 1.0, 1.6)], ["c = 0.5", "c = 1 (original)", "c = 1.6"], tamanho=4)

### ✍️ Tarefa 3.1 — Faixa de foco
🎯 **Objetivo:** gerar a **máscara de foco** da miniatura, uma faixa horizontal nítida com transição gradual para as regiões desfocadas.

<p align="center">
  <img src="https://i.ibb.co/R44DTgvQ/faixa-foco-1.png" width="320" alt="Game Boy original">
</p>

**O que é a máscara?** É uma imagem **2D**, com o mesmo número de linhas e colunas da foto, que indica **o quanto cada pixel deve permanecer nítido**. Vale 255 na faixa de foco, 0 longe dela e valores intermediários na transição. Ela será usada na Tarefa 3.3 como o **alfa** de uma composição. Os três parâmetros são frações da altura da foto. `centro` define a posição da faixa, como exemplo se `centro=0.6` a faixa deverá ser colocada a 60% da altura, contando a partir do topo. `largura` define a espessura da parte nítida, e `transicao` a extensão da passagem gradual para o desfoque. Esses valores devem ser **ajustados para cada foto**, de modo que a faixa nítida fique sobre o que deve estar em foco, como uma rua, um grupo de pessoas ou uma fileira de carros.

**Partindo do exemplo resolvido:** o contraste mostrou uma conta vetorizada em `float` seguida de `np.clip`. Aqui a ideia é a mesma, aplicada à **distância** entre cada linha e a faixa. Como a nitidez depende só da linha, basta calcular um valor por linha e repeti-lo em todas as colunas por **broadcasting**, como num gradiente.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_3_1 = imagem_da_tarefa("3.1", ENVIAR, padrao=img_teste)

def faixa_de_foco(img, centro=0.6, largura=0.15, transicao=0.25):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(faixa_de_foco, img_3_1)

### ✍️ Tarefa 3.2 — Desfoque por reamostragem
🎯 **Objetivo:** obter uma versão **desfocada** da imagem utilizando apenas `cv2.resize`.

<p align="center">
  <img src="https://i.ibb.co/1GK8mmts/desfocado.png" width="320" alt="Imagem desfocada">
</p>

**Por que reduzir e ampliar desfoca?** Ao reduzir a imagem, os detalhes finos são **descartados**. Ao ampliá-la de volta, a interpolação apenas **estima** os pixels que faltam. Com uma interpolação suave, como a bicúbica, essa estimativa resulta numa imagem com aparência desfocada.

**Partindo da Tarefa 1.2:** no filtro 8-bit vocês fizeram a mesma redução, mas ampliaram com vizinho mais próximo, o que produziu blocos. Trocar o método de interpolação troca o efeito. O parâmetro `fator` define o quanto a imagem é reduzida e `passadas` quantas vezes o processo é repetido. A saída deve ter as **mesmas dimensões** da entrada.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_3_2 = imagem_da_tarefa("3.2", ENVIAR, padrao=img_teste)

def desfoque_reamostragem(img, fator=6, passadas=2):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(desfoque_reamostragem, img_3_2)

### ✍️ Tarefa 3.3 — Miniatura (efeito *tilt-shift*)
🎯 **Objetivo:** combinar a imagem **nítida** na faixa de foco com a versão **desfocada** fora dela, com cores intensificadas, produzindo o efeito de miniatura.

<div align="center">
<table>
  <tr>
    <td align="center">
      <img src="https://i.ibb.co/k62bv4hq/urban-twilight-crossing-stockcake.jpg" width="320" alt="Imagem original"><br>
      <em>Imagem original</em>
    </td>
    <td align="center">
      <img src="https://i.ibb.co/rDvbRKt/mini.png" width="320" alt="Tilt-shift"><br>
      <em>Efeito Tilt-shift</em>
    </td>
  </tr>
</table>
</div>

**Como as peças se juntam?** Por uma **composição alfa**, `alfa * frente + (1 - alfa) * fundo`. A frente é a imagem nítida, o fundo é o desfoque da Tarefa 3.2 e o alfa é a máscara da Tarefa 3.1 convertida para a faixa de 0 a 1. Como maquetes costumam ter cores mais vivas que cenas reais, a imagem também recebe mais **saturação** e mais **contraste**.

**Partindo do exemplo resolvido:** a função `contraste` já está pronta. A saturação segue a mesma ideia, mas afasta cada cor do **cinza do próprio pixel** em vez de 128. Completem a função auxiliar `saturar(img, k)` com esse princípio.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_3_3 = imagem_da_tarefa("3.3", ENVIAR, padrao=img_teste)

def saturar(img, k=1.6):
    # SEU CÓDIGO AQUI
    raise NotImplementedError


def miniatura(img, centro=0.6, largura=0.15, transicao=0.25, fator=8):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(miniatura, img_3_3)

---
# 🔷 Etapa 4 — Caleidoscópio e simetria
Nesta etapa, vocês vão construir padrões simétricos a partir de uma região da imagem, aplicando **espelhamentos**, **transposição** de arrays e, na última tarefa, **coordenadas polares**. Imagens com cores e texturas variadas produzem padrões mais ricos.


### ✨ Exemplo resolvido — Espelho vertical
A **simetria** em relação a um eixo significa que um lado da imagem é o reflexo do outro. Este exemplo torna a imagem simétrica em relação ao **eixo vertical**: a metade esquerda é copiada, espelhada, sobre a metade direita.

A implementação segue três passos:
1. **Copiar a entrada** (`img.copy()`), para que a original não seja alterada.
2. **Selecionar a metade esquerda** com fatiamento (`img[:, :meio]`) e **espelhá-la** com passo -1 nas colunas (`[:, ::-1]`).
3. **Atribuir** o resultado à metade direita da cópia. Usar `w - meio` como início garante que os tamanhos coincidam, mesmo quando a largura é ímpar.

> 📚 Conceitos da *Aula Prática 01*: fatiamento e espelhamento (Partes 3.3 e 5.4) e visão × cópia (Parte 3.7).

O espelhamento com passo -1 é a base das duas primeiras tarefas desta etapa, e a ideia de reflexo continua na terceira.

In [ ]:
def espelho_vertical(img):
    """Espelha a metade esquerda sobre a direita.

    Conceitos: cópia, fatiamento com passo -1, shape para achar o meio (largura par ou ímpar).
    """
    saida = img.copy()
    w = img.shape[1]
    meio = w // 2
    saida[:, w - meio:] = img[:, :meio][:, ::-1]          # metade esquerda invertida
    return saida

_ = mostrar_resultado(espelho_vertical, img_teste)

### ✍️ Tarefa 4.1 — Simetria de 4
🎯 **Objetivo:** construir uma imagem simétrica em relação aos **dois eixos**, vertical e horizontal, a partir de um único quadrante.

<div align="center">
<table>
  <tr>
    <td align="center">
      <img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2f/Different_Slant_on_Orion_%28495636660%29.jpg/960px-Different_Slant_on_Orion_%28495636660%29.jpg" width="300" alt="Imagem original"><br>
      <em>Imagem original (Nebulosa de Órion)</em>
    </td>
    <td align="center">
      <img src="https://i.ibb.co/1tLPwBhb/simetria4.png" width="300" alt="Imagem com simetria de 4"><br>
      <em>Simetria de 4</em>
    </td>
  </tr>
</table>
</div>

**Como se forma a simetria?** A imagem final tem 4 quadrantes. O **superior esquerdo** vem da foto e os outros três são cópias espelhadas dele, uma na horizontal, uma na vertical e uma nas duas direções. O resultado é uma imagem quadrada de `lado × lado`.

**Partindo do exemplo resolvido:** o rosto simétrico espelhou uma metade com **passo -1** nas colunas. Aqui a mesma operação é feita também nas linhas, e os pedaços são unidos com `np.hstack` e `np.vstack`. Para deixar a imagem quadrada, reaproveitem a `recorte_quadrado` da Tarefa 2.3.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_4_1 = imagem_da_tarefa("4.1", ENVIAR, padrao=img_teste)

def simetria_4(img, lado=600):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(simetria_4, img_4_1)

### ✍️ Tarefa 4.2 — Caleidoscópio de 8
🎯 **Objetivo:** estender a simetria às **diagonais**, obtendo 8 setores equivalentes, como em um caleidoscópio.

<div align="center">
<table>
  <tr>
    <td align="center">
      <img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2f/Different_Slant_on_Orion_%28495636660%29.jpg/960px-Different_Slant_on_Orion_%28495636660%29.jpg" width="300" alt="Imagem original"><br>
      <em>Imagem original (Nebulosa de Órion)</em>
    </td>
    <td align="center">
      <img src="https://i.ibb.co/rKqQvfhc/simetria8.png" width="300" alt="Caledoscópio de 8"><br>
      <em>Caledoscópio de 8</em>
    </td>
  </tr>
</table>
</div>

**O que muda em relação à Tarefa 4.1?** Na 4.1, o quadrante era espelhado do jeito que vinha da foto. Aqui, antes disso, ele é dobrado na sua **diagonal**, a linha que liga o canto superior esquerdo ao canto inferior direito. A metade abaixo da diagonal é mantida, e a metade acima passa a ser o reflexo dela, como se o quadrante fosse dobrado nessa linha. Assim, o pixel da linha 2 e coluna 5 recebe a cor do pixel da linha 5 e coluna 2. Trocar linha por coluna é a **transposição**, que gera de uma vez esse reflexo para todos os pixels. Numa imagem colorida, só os dois primeiros eixos, linhas e colunas, são trocados, e os canais de cor ficam no lugar.

**Partindo da Tarefa 4.1:** o espelhamento final é o mesmo. A novidade é escolher, pixel a pixel, entre o quadrante original e o transposto conforme o pixel esteja acima ou abaixo da diagonal. Para saber quais pixels ficam acima da diagonal, usem as coordenadas de `np.mgrid`, como na figurinha. Acima da diagonal estão os pixels em que a coluna é maior que a linha.


In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_4_2 = imagem_da_tarefa("4.2", ENVIAR, padrao=img_teste)

def caleidoscopio_8(img, lado=600):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(caleidoscopio_8, img_4_2)

### ✍️ Tarefa 4.3 — Caleidoscópio radial
🎯 **Objetivo:** construir um caleidoscópio **circular**, em que uma fatia da imagem, em forma de pedaço de pizza, se repete e se espelha ao redor do centro.

<div align="center">
<table>
  <tr>
    <td align="center">
      <img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2f/Different_Slant_on_Orion_%28495636660%29.jpg/960px-Different_Slant_on_Orion_%28495636660%29.jpg" width="300" alt="Imagem original"><br>
      <em>Imagem original (Nebulosa de Órion)</em>
    </td>
    <td align="center">
      <img src="https://i.ibb.co/Z6PNGSvf/simetria-radial.png" width="300" alt="Caleidoscópio radial"><br>
      <em>Caleidoscópio radial</em>
    </td>
  </tr>
</table>
</div>

**Qual é a ideia?** Pense numa pizza cortada em 6 fatias iguais. Cada fatia ocupa 60° da volta completa de 360°. No caleidoscópio, todas as fatias mostram o mesmo desenho, e cada fatia é o reflexo da vizinha.

Para montar isso, cada pixel da imagem final faz uma pergunta: **"de qual pixel da foto eu copio a minha cor?"**. A resposta vem em três passos.

**Passo 1. Onde o pixel está em relação ao centro.** Todo pixel pode ser localizado por duas informações, como o ponteiro de um relógio que sai do centro e aponta para ele.
- A **distância** até o centro, que vocês já calcularam na figurinha (Tarefa 2.3).
- O **ângulo**, ou seja, para que lado o ponteiro aponta. 0° é para a direita, 90° é para baixo, 180° é para a esquerda e 270° é para cima.

O ângulo é calculado com `np.degrees(np.arctan2(dy, dx)) % 360`, em que `dx` e `dy` são as distâncias horizontal e vertical do pixel até o centro.

**Passo 2. Trazer o ângulo para a primeira fatia.** Como todas as fatias são iguais, basta saber em que ponto da sua fatia o pixel está. Para isso, usa-se o resto da divisão pelo tamanho da fatia. Um pixel a 130° fica a `130 % 60 = 10°` do início da sua fatia, então copia a cor do pixel a 10° da primeira fatia.

Para que as fatias vizinhas fiquem espelhadas, cada fatia é dividida ao meio. Se o ângulo cair na primeira metade (de 0° a 30°), ele continua igual. Se cair na segunda metade (de 30° a 60°), ele é trocado pelo seu reflexo. Um pixel a 40°, por exemplo, passa a usar `60 - 40 = 20°`. Essa troca pode ser feita com `np.where`.

**Passo 3. Achar o pixel da foto e copiar a cor.** Agora sabemos a distância e o novo ângulo. Falta transformar isso de volta em linha e coluna. As fórmulas são estas:

- `coluna = centro + distancia * np.cos(np.radians(angulo))`
- `linha = centro + distancia * np.sin(np.radians(angulo))`

Elas apenas fazem o caminho inverso do Passo 1. Arredondem os resultados para inteiros e usem `np.clip` para que fiquem dentro da imagem. Com os arrays de linhas e colunas prontos, `q[linhas, colunas]` copia a cor de todos os pixels de uma vez, sem laços.

A imagem `q` é o quadrado gerado pela `recorte_quadrado`. Os pixels fora do círculo de diâmetro `lado` ficam pretos.

In [ ]:
# 📤 Para usar outra imagem nesta tarefa, altere ENVIAR para True, execute e selecione o arquivo.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_4_3 = imagem_da_tarefa("4.3", ENVIAR, padrao=img_teste)

def caleidoscopio_radial(img, setores=6, lado=600):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

_ = mostrar_resultado(caleidoscopio_radial, img_4_3)


---
# 🕵️ Desafio — Esteganografia
Esta etapa é um **desafio opcional**. Ela usa operadores de bits, que vão além do que foi trabalhado na nossa *Aula Prática 01*, e por isso começa com uma explicação de como eles funcionam.

A esteganografia consiste em **ocultar informação dentro de outro conteúdo**, de forma imperceptível. Aqui, uma imagem inteira é escondida nos **bits menos significativos** dos pixels de outra. Alterar esses bits modifica cada valor muito pouco na escala de 0 a 255, uma diferença visualmente imperceptível.

### 🔢 Como funciona a manipulação de bits
Todo valor `uint8` é guardado como **8 bits**. O valor 181, por exemplo, é `10110101` em binário. Cada posição vale o dobro da posição à sua direita, da direita para a esquerda: 1, 2, 4, 8, 16, 32, 64 e 128. Assim, `10110101` = 128 + 32 + 16 + 4 + 1 = 181.

Os bits da **esquerda** são os **mais significativos**: mudar o primeiro bit altera o valor em 128. Os da **direita** são os **menos significativos**: mudar o último altera o valor em apenas 1. É por isso que dá para esconder informação nos bits da direita sem que a imagem mude visivelmente.

Os operadores de bits trabalham diretamente sobre essas posições:

| Operador | O que faz | Exemplo com 8 bits |
|---|---|---|
| `x >> n` | desloca os bits `n` posições para a **direita**. Os `n` bits da direita são descartados e entram zeros pela esquerda | `11001000 >> 6` → `00000011` |
| `x << n` | desloca os bits `n` posições para a **esquerda**. Entram zeros pela direita | `00000011 << 6` → `11000000` |
| `x & m` | **E** bit a bit. O resultado tem 1 só onde `x` **e** `m` têm 1. Serve para **zerar** ou **isolar** posições | `10110101 & 11111100` → `10110100` |
| `x \| m` | **OU** bit a bit. O resultado tem 1 onde `x` **ou** `m` têm 1. Serve para **ligar** posições | `10110100 \| 00000011` → `10110111` |

O número `m` usado com `&` é chamado de **máscara**: os 1 indicam as posições que serão mantidas, e os 0, as que serão zeradas. Com 2 bits, as duas máscaras desta etapa são `11111100` (mantém os 6 bits altos) e `00000011` (mantém os 2 bits baixos). Em Python, elas podem ser escritas como `0b11111100` e `0b00000011`.

**Um exemplo completo com 2 bits.** Vamos esconder o pixel do segredo de valor 200 (`11001000`) no pixel da capa de valor 181 (`10110101`).
1. `200 >> 6` traz os 2 bits mais altos do segredo para o final, resultando em `00000011`.
2. `181 & 0b11111100` zera os 2 bits mais baixos da capa, resultando em `10110100`.
3. `|` junta as duas partes, resultando em `10110111`, que vale **183**. A capa mudou de 181 para 183, uma diferença imperceptível.

Para revelar, faz-se o caminho inverso.
1. `183 & 0b00000011` isola os 2 bits mais baixos, resultando em `00000011`.
2. `<< 6` devolve esses bits para o início, resultando em `11000000`, que vale **192**.

O segredo volta como 192 em vez de 200, porque só os 2 bits mais altos foram guardados. É a mesma perda da quantização em 4 níveis, e a imagem continua reconhecível. Em NumPy, essas operações são aplicadas a todos os pixels de uma vez, sem laços. A célula abaixo reproduz o exemplo.


In [ ]:
# 🔢 Operadores de bits no exemplo acima (np.binary_repr mostra o valor em binário)
capa, segredo = np.uint8(181), np.uint8(200)
for nome, valor in [("capa", capa),
                    ("segredo", segredo),
                    ("segredo >> 6", segredo >> 6),
                    ("capa & 0b11111100", capa & 0b11111100),
                    ("escondido", (capa & 0b11111100) | (segredo >> 6)),
                    ("revelado", (((capa & 0b11111100) | (segredo >> 6)) & 0b00000011) << 6)]:
    print(f"{nome:<20} {np.binary_repr(int(valor) & 0xFF, width=8)}  = {int(valor) & 0xFF}")


### ✨ Exemplo resolvido — Planos de bits
Cada valor `uint8` tem **8 bits**, do bit 7 (o mais significativo, que vale 128) ao bit 0 (o menos significativo, que vale 1). O **plano de bits** `k` é a imagem formada apenas pelo bit `k` de cada pixel.

A implementação usa dois dos operadores apresentados acima:
1. **Deslocamento para a direita** (`canal >> bit`): traz o bit desejado para a última posição.
2. **E bit a bit** (`& 1`): mantém apenas o último bit, que vale 0 ou 1.
Multiplicar por 255 transforma o resultado em preto e branco, para exibição.

Observe os resultados: os planos **mais significativos** (7 e 6) mostram claramente a estrutura da imagem, enquanto os **menos significativos** (1 e 0) se assemelham a **ruído**. Alterar esses bits não muda a aparência da imagem, e é neles que a informação será ocultada.

> 📚 Conceitos da *Aula Prática 01*: o tipo `uint8` e seus 8 bits (Parte 3.2) e a relação entre bits e níveis de intensidade (Parte 6.5).

In [ ]:
def plano_de_bits(img, bit=7):
    """Imagem (0 ou 255) com o bit `bit` de cada pixel do canal verde.

    Conceitos: operadores bit a bit (>>, &), conversão para exibição.
    """
    canal = img[:, :, 1]
    return (((canal >> bit) & 1) * 255).astype(np.uint8)

mostrar_varias([plano_de_bits(img_teste, b) for b in (7, 6, 4, 2, 1, 0)],
               [f"bit {b}" for b in (7, 6, 4, 2, 1, 0)], colunas=6, tamanho=2.5)

### ✍️ Tarefa 5.1 — Esconder uma imagem dentro de outra
🎯 **Objetivo:** ocultar uma **imagem completa**, o segredo, dentro de outra, a capa.

**Como uma imagem inteira pode ser escondida?** Na quantização, vimos que uma imagem com apenas 2 bits por canal ainda é reconhecível, porque os **bits mais significativos** carregam quase toda a informação visual. Já os **bits menos significativos** da capa podem ser alterados sem mudança perceptível. A técnica consiste em substituir os `bits` bits menos significativos da capa pelos `bits` bits mais significativos do segredo, redimensionado para o tamanho da capa.

**Partindo do exemplo resolvido e da explicação de bits:** o plano de bits usou `>>` para trazer um bit para a última posição. Aqui o mesmo deslocamento traz os bits altos do segredo para as posições baixas. Na capa, uma máscara com `&` zera as posições baixas, e as duas partes se juntam com `|`, exatamente como no exemplo do pixel 181. A diferença é que a quantidade de bits é o parâmetro `bits`, então a máscara também precisa ser calculada a partir dele, e o segredo deve ser redimensionado para o tamanho da capa. Comparem o resultado com `bits` igual a 1, 2, 3 e 4. A partir de qual valor a alteração na capa se torna perceptível?


In [ ]:
# 📤 Imagem de CAPA: altere ENVIAR para True para selecionar outra.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_5_1 = imagem_da_tarefa("5.1", ENVIAR, padrao=img_teste)

# 📤 Imagem a ser OCULTADA: mesmo procedimento, com ENVIAR_B.
ENVIAR_B = False
SEGREDO = imagem_da_tarefa("5.1b", ENVIAR_B, padrao=cv2.cvtColor(sk_data.astronaut(), cv2.COLOR_RGB2BGR))

def esconder_imagem(capa, segredo, bits=2):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

capa_com_segredo = mostrar_resultado(esconder_imagem, img_5_1, SEGREDO)
if capa_com_segredo is not None:
    cv2.imwrite("resultados/capa_com_segredo.png", capa_com_segredo)    # arquivo PNG a ser trocado entre as equipes

### ✍️ Tarefa 5.2 — Revelar a imagem e o teste do JPEG
🎯 **Objetivo:** recuperar a imagem oculta na Tarefa 5.1 e analisar por que a compressão **JPEG** destrói a informação escondida.

**Por que o formato do arquivo importa?** O **PNG** usa compressão **sem perdas**, então ao abrir o arquivo os pixels são exatamente os mesmos que foram salvos. O **JPEG** usa compressão **com perdas**, e os valores sofrem pequenas alterações, imperceptíveis a olho nu, mas que atingem justamente os bits menos significativos.

**Partindo da Tarefa 5.1:** é o caminho inverso. Isolem os bits baixos da capa com uma máscara e devolvam-nos às posições altas com `<<`. O segredo volta com poucos níveis por canal, já que só `bits` bits de cada valor foram guardados. Depois, executem a célula de teste abaixo da função e **expliquem no README** o que aconteceu com a imagem escondida em cada formato.

In [ ]:
# 📤 Para revelar uma capa recebida de outra equipe, altere ENVIAR para True e envie o arquivo PNG.
#    A imagem permanece armazenada; após o envio, retorne ENVIAR para False.
ENVIAR = False
img_5_2 = imagem_da_tarefa("5.2", ENVIAR, padrao=capa_com_segredo, reduzir=False, usar_foto_teste=False)

def revelar_imagem(img, bits=2):
    # SEU CÓDIGO AQUI
    raise NotImplementedError

if img_5_2 is not None:
    _ = mostrar_resultado(revelar_imagem, img_5_2)
else:
    print("⏳ faça a Tarefa 5.1 primeiro (ou envie uma capa com segredo pela linha 📤).")

In [ ]:
# 🧪 Teste do JPEG × PNG: salvamos a capa com segredo nos dois formatos e tentamos revelar de novo
try:
    if capa_com_segredo is None:
        raise NotImplementedError
    for ext in (".png", ".jpg"):
        volta = cv2.imdecode(cv2.imencode(ext, capa_com_segredo)[1], cv2.IMREAD_COLOR)
        mostrar(revelar_imagem(volta), f"imagem secreta depois de salvar em {ext}", tamanho=4)
except NotImplementedError:
    print("⏳ termine as tarefas 5.1 e 5.2 para rodar o teste.")


---
# 🎁 Bônus — Interface com Gradio
Disponibiliza as funções implementadas em uma interface web simples, com envio de imagem e seleção do filtro. Altere `USAR_GRADIO` para `True` e execute; no Colab, a interface é exibida abaixo da célula. São incluídas as funções que recebem **uma** imagem e já estão implementadas.

In [ ]:
USAR_GRADIO = False

if USAR_GRADIO:
    os.system("pip install -q gradio")
    import gradio as gr

    candidatos = [filtro_fosforo_verde, filtro_gameboy, filtro_8bit, caixa_de_dialogo,
                  faixa_de_legenda, meme_classico, zoom_dramatico, contraste, miniatura,
                  espelho_vertical, simetria_4, caleidoscopio_8, caleidoscopio_radial]

    FILTROS_APP = {}
    amostra = cv2.resize(img_teste, (64, 64))
    for f in candidatos:                     # mantém só os filtros que já funcionam
        try:
            if f(amostra) is not None:
                FILTROS_APP[f.__name__] = f
        except Exception:
            pass

    def aplicar(img_rgb, nome_filtro):
        bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        saida = FILTROS_APP[nome_filtro](limitar_tamanho(bgr))
        return saida if saida.ndim == 2 else cv2.cvtColor(saida, cv2.COLOR_BGR2RGB)

    gr.Interface(
        fn=aplicar,
        inputs=[gr.Image(type="numpy", label="Sua imagem"),
                gr.Dropdown(list(FILTROS_APP), value=list(FILTROS_APP)[0], label="Filtro")],
        outputs=gr.Image(label="Resultado"),
        title="FiltroLab — nome da equipe",
    ).launch()